# 05 — Ablation Study: Comparing LoRA Ranks r=4, r=8, r=16

**Project**: LLM Fine-Tuning with QLoRA  
**Notebook**: 5 of 6  
**GPU required**: ✅ Yes — T4  
**Runtime**: ~90–120 minutes (3 full training runs)

---

## What This Notebook Does

An **ablation study** is a controlled experiment where you change ONE variable
and measure the effect. This is exactly how ML researchers and engineers think.

We train 3 models — identical in every way except the LoRA rank `r`:

| Run | LoRA Rank | Trainable Params | Hypothesis |
|---|---|---|---|
| A | r = 4  | ~2M  | Underfits — too little adapter capacity |
| B | r = 8  | ~4M  | Balanced — our main model |
| C | r = 16 | ~8M  | Most capacity — may not improve further |

Then we compare: ROUGE scores, training time, and trainable parameter count.

## Why This Matters for Your Portfolio

> Most portfolios show ONE experiment. An ablation study shows you can think like
> a researcher — designing controlled experiments to understand your system.
> This is a skill that's rare and highly valued in ML roles.

---
## ⚙️ Cell 1: Setup

In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate evaluate rouge-score matplotlib pyyaml
print("✅ Dependencies installed")

In [ ]:
import os, sys, json, time
import torch
import matplotlib.pyplot as plt
import numpy as np

GITHUB_USERNAME = "YOUR_USERNAME"   # ← change this
REPO_NAME = "llm-finetuning-peft"

if not os.path.exists(REPO_NAME):
    !git clone https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
else:
    !cd {REPO_NAME} && git pull

sys.path.insert(0, f"/content/{REPO_NAME}/src")
os.chdir(f"/content/{REPO_NAME}")
os.makedirs("results", exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("❌ GPU required.")
print(f"✅ GPU: {torch.cuda.get_device_name(0)} | {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB VRAM")

---
## 📐 Cell 2: Understanding LoRA Rank — The Variable Being Tested

Before running experiments, let's understand exactly what changing `r` does.

In [ ]:
from model_utils import get_lora_config, load_tokenizer, load_base_model, apply_lora, print_trainable_parameters

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
RANKS = [4, 8, 16]

# ── Show how rank affects parameter count WITHOUT loading the full model ───
# Calculate theoretical adapter size for each rank
hidden_size = 2048    # TinyLlama hidden dimension
num_layers  = 22      # TinyLlama transformer layers
num_modules = 4       # q, k, v, o projections

print("📐 LoRA Parameter Count by Rank")
print("=" * 65)
print(f"  {'Rank':>6} | {'A matrix':>15} | {'B matrix':>15} | {'Total (all layers)':>20}")
print("  " + "-" * 62)

for r in RANKS:
    # Each LoRA layer: A is (hidden x r), B is (r x hidden)
    params_per_module = (hidden_size * r) + (r * hidden_size)
    total = params_per_module * num_modules * num_layers
    print(f"  r={r:<4} | {hidden_size}×{r}={hidden_size*r:>6,} | {r}×{hidden_size}={r*hidden_size:>6,} | {total:>18,} params")

print()
print(f"  Base model total params: ~1,100,048,384")
for r in RANKS:
    params_per_module = (hidden_size * r) + (r * hidden_size)
    total = params_per_module * num_modules * num_layers
    pct = total / 1_100_048_384 * 100
    print(f"  r={r}: {pct:.3f}% of base model parameters are trainable")

---
## 🔬 Cell 3: Load Dataset (Once — Reused Across All Runs)

In [ ]:
from data_utils import load_alpaca_dataset

# Use a smaller subset for the ablation — 1000 train samples to keep each run shorter
# This is fine for comparing ranks — absolute performance matters less than relative comparison
ABLATION_TRAIN_SIZE = 1000
ABLATION_TEST_SIZE  = 100

train_dataset, test_dataset = load_alpaca_dataset(
    train_size=ABLATION_TRAIN_SIZE,
    test_size=ABLATION_TEST_SIZE,
    seed=42
)

print(f"✅ Ablation dataset ready: {len(train_dataset)} train / {len(test_dataset)} test")
print(f"   (Smaller than main run to keep each ablation training under ~15 min)")

---
## 🏋️ Cell 4: Run All 3 Training Experiments

⏱️ **Total estimated time: 90–120 minutes** (~30–40 min per rank)

Each run is identical except for the LoRA rank `r`.  
After each run, we save the adapter and record metrics.

In [ ]:
import yaml
from train import get_training_args, load_config
from trl import SFTTrainer
from evaluate import compute_rouge_scores

config = load_config("configs/qlora_config.yaml")
config["dataset"]["train_size"] = ABLATION_TRAIN_SIZE
config["dataset"]["test_size"]  = ABLATION_TEST_SIZE

# Storage for results across all runs
ablation_results = []

for rank in RANKS:
    print()
    print("★" * 60)
    print(f"  ABLATION RUN: LoRA rank r={rank}")
    print("★" * 60)

    # ── Load fresh base model for each run ────────────────────────────────
    tokenizer = load_tokenizer(MODEL_NAME)
    model     = load_base_model(MODEL_NAME)

    # ── Apply LoRA with this rank ──────────────────────────────────────────
    lora_config = get_lora_config(r=rank)
    model = apply_lora(model, lora_config)
    print_trainable_parameters(model)

    # ── Configure training ─────────────────────────────────────────────────
    training_args = get_training_args(config)
    training_args.output_dir = f"outputs/ablation-r{rank}"
    os.makedirs(training_args.output_dir, exist_ok=True)

    # ── Train ──────────────────────────────────────────────────────────────
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        dataset_text_field="text",
        max_seq_length=config["model"]["max_seq_length"],
        args=training_args,
    )

    start_time = time.time()
    result = trainer.train()
    training_time = (time.time() - start_time) / 60

    # ── Evaluate ───────────────────────────────────────────────────────────
    model.eval()
    rouge = compute_rouge_scores(model, tokenizer, test_dataset, num_samples=50)

    # Count trainable parameters
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # ── Extract loss logs ──────────────────────────────────────────────────
    logs   = trainer.state.log_history
    steps  = [l["step"] for l in logs if "loss" in l]
    losses = [l["loss"]  for l in logs if "loss" in l]

    # ── Save adapter ───────────────────────────────────────────────────────
    adapter_path = f"outputs/ablation-adapter-r{rank}"
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)

    # ── Store results ──────────────────────────────────────────────────────
    run_result = {
        "rank": rank,
        "trainable_params": trainable_params,
        "final_loss": round(result.training_loss, 4),
        "training_time_min": round(training_time, 1),
        "rouge1": round(rouge["rouge1"], 4),
        "rouge2": round(rouge["rouge2"], 4),
        "rougeL": round(rouge["rougeL"], 4),
        "loss_history": {"steps": steps, "losses": losses},
    }
    ablation_results.append(run_result)

    print(f"\n✅ r={rank} complete:")
    print(f"   Trainable params: {trainable_params:,}")
    print(f"   Final loss:       {run_result['final_loss']}")
    print(f"   Training time:    {training_time:.1f} min")
    print(f"   ROUGE-L:          {run_result['rougeL']}")

    # Free GPU memory before next run
    del model, trainer
    torch.cuda.empty_cache()

print("\n" + "=" * 60)
print("  ALL 3 ABLATION RUNS COMPLETE!")
print("=" * 60)

---
## 📊 Cell 5: Results Table + Analysis

In [ ]:
print("📊 ABLATION STUDY RESULTS")
print("=" * 80)
print(f"  {'Rank':>6} | {'Trainable Params':>18} | {'Final Loss':>12} | {'Time (min)':>12} | {'ROUGE-1':>9} | {'ROUGE-L':>9}")
print("  " + "-" * 75)

for r in ablation_results:
    print(f"  r={r['rank']:<4} | {r['trainable_params']:>18,} | {r['final_loss']:>12.4f} | "
          f"{r['training_time_min']:>12.1f} | {r['rouge1']:>9.4f} | {r['rougeL']:>9.4f}")

print()
print("💡 Key Questions to Answer:")
print("   1. Does higher rank always mean better ROUGE? (Diminishing returns?)")
print("   2. How much more time does r=16 take vs r=4?")
print("   3. Which rank gives the best tradeoff between quality and efficiency?")
print("   4. Is the jump from r=4 to r=8 bigger than r=8 to r=16?")

In [ ]:
# ── Save ablation results to JSON ─────────────────────────────────────────
# Remove loss_history from saved JSON (too large) — keep it only for plotting
save_data = [
    {k: v for k, v in r.items() if k != "loss_history"}
    for r in ablation_results
]

with open("results/ablation_results.json", "w") as f:
    json.dump(save_data, f, indent=2)

print("💾 Ablation results saved to results/ablation_results.json")

---
## 📈 Cell 6: Visualization — 3-Panel Ablation Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Ablation Study: LoRA Rank Comparison (TinyLlama-1.1B on Alpaca)",
             fontsize=14, fontweight="bold", y=1.02)

ranks_list  = [r["rank"] for r in ablation_results]
colors      = ["#94A3B8", "#4F46E5", "#10B981"]
rank_labels = [f"r={r}" for r in ranks_list]

# ── Panel 1: ROUGE-L comparison ────────────────────────────────────────────
ax = axes[0]
rougeL_vals = [r["rougeL"] for r in ablation_results]
bars = ax.bar(rank_labels, rougeL_vals, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, rougeL_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("ROUGE-L Score", fontsize=12, fontweight="bold")
ax.set_ylabel("ROUGE-L")
ax.set_ylim(0, max(rougeL_vals) * 1.2)
ax.grid(True, axis="y", alpha=0.3)

# ── Panel 2: Training time comparison ─────────────────────────────────────
ax = axes[1]
time_vals = [r["training_time_min"] for r in ablation_results]
bars = ax.bar(rank_labels, time_vals, color=colors, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, time_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f} min", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Training Time", fontsize=12, fontweight="bold")
ax.set_ylabel("Minutes")
ax.set_ylim(0, max(time_vals) * 1.2)
ax.grid(True, axis="y", alpha=0.3)

# ── Panel 3: Loss curves overlay ──────────────────────────────────────────
ax = axes[2]
for r, color in zip(ablation_results, colors):
    steps  = r["loss_history"]["steps"]
    losses = r["loss_history"]["losses"]
    # Smooth for clarity
    w = max(3, len(losses) // 15)
    smoothed = np.convolve(losses, np.ones(w)/w, mode="valid")
    ax.plot(steps[w-1:], smoothed, color=color, linewidth=2.5, label=f"r={r['rank']}")

ax.set_title("Loss Curves (Smoothed)", fontsize=12, fontweight="bold")
ax.set_xlabel("Step")
ax.set_ylabel("Training Loss")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

for ax in axes:
    ax.set_facecolor("#F8FAFC")

plt.tight_layout()
plt.savefig("results/ablation_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📈 Ablation chart saved to results/ablation_results.png")

---
## 🧠 Cell 7: Interpret the Results

In [ ]:
# ── Auto-generate interpretation based on results ─────────────────────────
best_rouge = max(ablation_results, key=lambda x: x["rougeL"])
fastest    = min(ablation_results, key=lambda x: x["training_time_min"])
r4 = ablation_results[0]
r8 = ablation_results[1]
r16= ablation_results[2]

r4_to_r8  = (r8["rougeL"]  - r4["rougeL"])  / max(r4["rougeL"],  1e-8) * 100
r8_to_r16 = (r16["rougeL"] - r8["rougeL"])  / max(r8["rougeL"],  1e-8) * 100
time_r4_r16_pct = (r16["training_time_min"] - r4["training_time_min"]) / r4["training_time_min"] * 100

print("🧠 Ablation Study Interpretation")
print("=" * 60)
print()
print(f"Best ROUGE-L      : r={best_rouge['rank']} ({best_rouge['rougeL']:.4f})")
print(f"Fastest training  : r={fastest['rank']} ({fastest['training_time_min']:.1f} min)")
print()
print(f"ROUGE-L improvement r=4 → r=8  : {r4_to_r8:+.1f}%")
print(f"ROUGE-L improvement r=8 → r=16 : {r8_to_r16:+.1f}%")
print(f"Training time increase r=4→r=16: {time_r4_r16_pct:+.1f}%")
print()
print("📝 What This Tells Us:")
if abs(r8_to_r16) < 2:
    print("   ✅ Diminishing returns: r=8 and r=16 perform similarly")
    print("   ✅ r=8 is the sweet spot — best efficiency/quality tradeoff")
elif r8_to_r16 > 5:
    print("   📈 r=16 meaningfully outperforms r=8 — more rank helps here")
    print("   📌 Consider using r=16 if compute allows")
print()
print("💼 Portfolio talking point:")
print("   'I conducted an ablation study varying LoRA rank and found that")
print(f"   r=8 provided the best tradeoff, achieving {r8['rougeL']:.3f} ROUGE-L")
print(f"   while r=16 added {time_r4_r16_pct:.0f}% training overhead for minimal gain.'")

---
## 📤 Cell 8: Commit Results

In [ ]:
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
!git add results/ablation_results.json results/ablation_results.png
!git commit -m "results: add ablation study - LoRA rank r=4/8/16 comparison"
!git push
print("✅ Ablation results committed!")

---
## ✅ Summary

In this notebook you ran a proper controlled experiment:

| What varied | LoRA rank: r = 4, 8, 16 |
|---|---|
| What was constant | Model, dataset, epochs, LR, all other hyperparams |
| What was measured | ROUGE-L, training time, trainable params |
| Output | 3-panel comparison chart + ablation_results.json |

### Key Concept: Diminishing Returns in LoRA
Doubling the rank (r=8 → r=16) doubles the adapter parameters but often yields
minimal ROUGE improvement. This is a common pattern — model capacity isn't the
bottleneck at small data sizes. Data quality and quantity matter more.

---
### ▶️ Next: `06_gradio_demo.ipynb`
Build an interactive web UI — the most visually impressive deliverable of this project.